# Árvores de Decisão Clássicas (ID3 / CART) no Diagnóstico de Falhas da HRS
## Estação de Reabastecimento de Hidrogênio (HRS) • SCADA-Core (ECAA08)

Este notebook implementa do zero em Python um construtor de **Árvore de Decisão Binária** com suporte aos dois critérios clássicos:
- **Entropia / Ganho de Informação** (base do ID3/C4.5);
- **Impureza de Gini** (base do CART).

Aplicamos o modelo para diagnosticar automaticamente 4 regimes operacionais da HRS (`Nominal`, `Falha_Chiller_Termica`, `Sobrepressao_Cascata`, `Vazamento_H2`) a partir da telemetria dos sensores `PT-101`, `TT-101`, `AT-101` e `PT-201`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from typing import Optional, Dict, Any, Tuple

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

### 1. Geração do Dataset Sintético de Diagnóstico da Estação de Hidrogênio

In [ ]:
def gerar_dataset_diagnostico_hrs(n_por_classe: int = 150) -> pd.DataFrame:
    registros = []
    # 1. Nominal
    for _ in range(n_por_classe):
        registros.append([
            np.random.normal(340, 15), np.random.normal(25, 8),
            np.abs(np.random.normal(2, 1.5)), np.random.normal(720, 30), "Nominal"
        ])
    # 2. Falha Chiller / Sobreaquecimento
    for _ in range(n_por_classe):
        registros.append([
            np.random.normal(350, 15), np.random.normal(76, 7),
            np.abs(np.random.normal(3, 1.5)), np.random.normal(740, 35), "Falha_Chiller_Termica"
        ])
    # 3. Sobrepressao Cascata
    for _ in range(n_por_classe):
        registros.append([
            np.random.normal(405, 14), np.random.normal(38, 9),
            np.abs(np.random.normal(3, 1.5)), np.random.normal(885, 25), "Sobrepressao_Cascata"
        ])
    # 4. Vazamento H2
    for _ in range(n_por_classe):
        registros.append([
            np.random.normal(325, 20), np.random.normal(32, 10),
            np.random.uniform(18, 65), np.random.normal(700, 40), "Vazamento_H2"
        ])
    df = pd.DataFrame(registros, columns=["PT_101_bar", "TT_101_degC", "AT_101_pctLFL", "PT_201_bar", "Classe"])
    return df.sample(frac=1.0, random_state=42).reset_index(drop=True)

df = gerar_dataset_diagnostico_hrs(150)
print(f"Dataset gerado: {len(df)} amostras | Classes: {df['Classe'].value_counts().to_dict()}")
display(df.head())

### 2. Implementação Pura de Entropia, Gini e Construtor Recursivo de Árvore (CART / ID3)

In [ ]:
def entropia_shannon(y: np.ndarray) -> float:
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return float(-np.sum(probs * np.log2(probs + 1e-12)))

def impureza_gini(y: np.ndarray) -> float:
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return float(1.0 - np.sum(probs ** 2))

class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, impurity=0.0, samples=0):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.impurity = impurity
        self.samples = samples

class ArvoreDecisaoClassica:
    def __init__(self, criterion: str = "gini", max_depth: int = 4, min_samples_split: int = 5):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root: Optional[TreeNode] = None
        self.feature_names = []

    def _impureza(self, y: np.ndarray) -> float:
        return impureza_gini(y) if self.criterion == "gini" else entropia_shannon(y)

    def _melhor_divisao(self, X: np.ndarray, y: np.ndarray) -> Tuple[Optional[int], Optional[float], float]:
        n_samples, n_features = X.shape
        imp_pai = self._impureza(y)
        melhor_ganho = -1.0
        melhor_feat, melhor_thresh = None, None

        for feat in range(n_features):
            valores = np.unique(X[:, feat])
            limiares = (valores[:-1] + valores[1:]) / 2.0
            for thresh in limiares:
                mask_esq = X[:, feat] <= thresh
                n_esq = int(np.sum(mask_esq))
                n_dir = n_samples - n_esq
                if n_esq == 0 or n_dir == 0:
                    continue
                imp_filhos = (n_esq / n_samples) * self._impureza(y[mask_esq]) + (n_dir / n_samples) * self._impureza(y[~mask_esq])
                ganho = imp_pai - imp_filhos
                if ganho > melhor_ganho:
                    melhor_ganho = ganho
                    melhor_feat = feat
                    melhor_thresh = float(thresh)
        return melhor_feat, melhor_thresh, melhor_ganho

    def _construir(self, X: np.ndarray, y: np.ndarray, depth: int) -> TreeNode:
        imp = self._impureza(y)
        classe_majoritaria = Counter(y).most_common(1)[0][0]

        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return TreeNode(value=classe_majoritaria, impurity=imp, samples=len(y))

        feat, thresh, ganho = self._melhor_divisao(X, y)
        if feat is None or ganho <= 1e-6:
            return TreeNode(value=classe_majoritaria, impurity=imp, samples=len(y))

        mask_esq = X[:, feat] <= thresh
        left = self._construir(X[mask_esq], y[mask_esq], depth + 1)
        right = self._construir(X[~mask_esq], y[~mask_esq], depth + 1)
        return TreeNode(feature=feat, threshold=thresh, left=left, right=right, impurity=imp, samples=len(y))

    def fit(self, X: np.ndarray, y: np.ndarray, feature_names=None):
        self.feature_names = feature_names or [f"X{i}" for i in range(X.shape[1])]
        self.root = self._construir(X, y, depth=0)

    def _predict_one(self, x: np.ndarray, node: TreeNode):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self._predict_one(x, self.root) for x in X])

    def imprimir_regras(self, node: Optional[TreeNode] = None, indent: str = ""):
        if node is None:
            node = self.root
        if node.value is not None:
            print(f"{indent}-> DIAGNÓSTICO: {node.value} (n={node.samples}, imp={node.impurity:.3f})")
            return
        nome_f = self.feature_names[node.feature]
        print(f"{indent}SE {nome_f} <= {node.threshold:.2f} (n={node.samples}, imp={node.impurity:.3f}):")
        self.imprimir_regras(node.left, indent + "  ")
        print(f"{indent}SENÃO ({nome_f} > {node.threshold:.2f}):")
        self.imprimir_regras(node.right, indent + "  ")

### 3. Treinamento, Comparação (Gini vs. Entropia) e Regras Extraídas

In [ ]:
features = ["PT_101_bar", "TT_101_degC", "AT_101_pctLFL", "PT_201_bar"]
X = df[features].values
y = df["Classe"].values

split = int(0.75 * len(df))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

for crit in ["gini", "entropy"]:
    clf = ArvoreDecisaoClassica(criterion=crit, max_depth=3)
    clf.fit(X_train, y_train, feature_names=features)
    preds = clf.predict(X_test)
    acc = np.mean(preds == y_test) * 100
    print(f"=== Critério: {crit.upper()} | Acurácia em Teste: {acc:.2f}% ===")
    if crit == "gini":
        clf.imprimir_regras()
        matriz = pd.crosstab(pd.Series(y_test, name="Real"), pd.Series(preds, name="Predito"))
        print("\nMatriz de Confusão (CART / Gini):")
        display(matriz)

### 4. Visualização das Fronteiras de Decisão no Plano `PT-101` $\times$ `TT-101`

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
cores = {
    "Nominal": "green",
    "Falha_Chiller_Termica": "orange",
    "Sobrepressao_Cascata": "red",
    "Vazamento_H2": "purple",
}
for classe, cor in cores.items():
    sub = df[df["Classe"] == classe]
    ax.scatter(sub["PT_101_bar"], sub["TT_101_degC"], c=cor, label=classe, alpha=0.7, s=28)

ax.axvline(380.0, color="red", linestyle="--", linewidth=1.3, label="Corte CART PT-101 (~380 bar)")
ax.axhline(55.0, color="orange", linestyle=":", linewidth=1.3, label="Corte CART TT-101 (~55 °C)")
ax.set_title("Separação de Regimes Operacionais da HRS por Árvore de Decisão")
ax.set_xlabel("Pressão PT-101 [bar]")
ax.set_ylabel("Temperatura TT-101 [°C]")
ax.legend()
plt.tight_layout()
plt.show()